# Ensemble Classification (Random Forest) Practical — Simple Exam Notebook

**Scenario:** Diabetes Outcome Prediction  
**Dataset:** `diabetes.csv`  
**Target:** `Diagnosis` (`0` = Negative, `1` = Positive)

## What we will do
1. Load and understand the dataset.
2. Explore missing values, target balance, and summary statistics.
3. Build RandomForestClassifier without PCA and tune `n_estimators`, `max_depth` with 5-fold Stratified CV.
4. Build RandomForestClassifier with PCA and tune similarly.
5. Compare both models and run paired t-test (`ttest_rel`).

**Important:** Keep `diabetes.csv` in the same directory as this notebook.


## 1. Import the tools

Exact cheatsheet imports: pandas, numpy, matplotlib, seaborn, StandardScaler, Pipeline, RandomForestClassifier, PCA, StratifiedKFold, GridSearchCV, metrics, and ttest_rel.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from scipy.stats import ttest_rel


## 2. Load the dataset and inspect it


In [ ]:
df = pd.read_csv("diabetes.csv")

print("First five rows:")
display(df.head())

print("Dataset shape:", df.shape)
print("\nColumn information:")
df.info()

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nNumerical summary:")
display(df.describe())


## 3. Explore Target Distribution


In [ ]:
target_col = "Diagnosis"

print("Class counts:")
print(df[target_col].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target_col)
plt.title("Target Distribution (0 = Negative, 1 = Positive)")
plt.show()


## 4. Separate Predictors (X) and Target (y)


In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

print("Features shape:", X.shape)
print("Target shape:", y.shape)


## 5. Build Random Forest without PCA and tune using GridSearchCV


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_no_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(random_state=42))
])

param_grid_no_pca = {
    "rf__n_estimators": [50, 100, 200],
    "rf__max_depth": [3, 5, 7, None],
    "rf__criterion": ["gini", "entropy"]
}

grid_no_pca = GridSearchCV(
    estimator=pipe_no_pca,
    param_grid=param_grid_no_pca,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_no_pca.fit(X, y)

print("Best Parameters (No PCA):", grid_no_pca.best_params_)
print("Best CV Accuracy (No PCA):", round(grid_no_pca.best_score_, 4))


## 6. Build Random Forest with PCA and tune using GridSearchCV


In [ ]:
pipe_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("rf", RandomForestClassifier(random_state=42))
])

param_grid_pca = {
    "rf__n_estimators": [50, 100, 200],
    "rf__max_depth": [3, 5, 7, None],
    "rf__criterion": ["gini", "entropy"]
}

grid_pca = GridSearchCV(
    estimator=pipe_pca,
    param_grid=param_grid_pca,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_pca.fit(X, y)

print("Best Parameters (With PCA):", grid_pca.best_params_)
print("Best CV Accuracy (With PCA):", round(grid_pca.best_score_, 4))


## 7. Compare Models & Statistical Significance Test (Paired t-test)


In [ ]:
best_no_pca = grid_no_pca.best_estimator_
best_pca = grid_pca.best_estimator_

scores_no_pca = cross_val_score(best_no_pca, X, y, cv=cv, scoring="accuracy")
scores_pca = cross_val_score(best_pca, X, y, cv=cv, scoring="accuracy")

print("Fold Scores (No PCA):", np.round(scores_no_pca, 4))
print("Mean Accuracy (No PCA):", round(scores_no_pca.mean(), 4))
print("\nFold Scores (With PCA):", np.round(scores_pca, 4))
print("Mean Accuracy (With PCA):", round(scores_pca.mean(), 4))

t_stat, p_val = ttest_rel(scores_no_pca, scores_pca)
print(f"\nPaired t-test: t-statistic = {t_stat:.4f}, p-value = {p_val:.4f}")

if p_val < 0.05:
    print("Significant difference between models (p < 0.05)")
else:
    print("No significant difference between models (p >= 0.05)")


## 8. Evaluation & Feature Importance (Cheatsheet Visualization)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

best_no_pca.fit(X_train, y_train)
y_pred = best_no_pca.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

# Feature importances from cheatsheet: plt.barh(<feature_names>, <importance>)
rf_model = best_no_pca.named_steps["rf"]
importances = rf_model.feature_importances_

plt.figure(figsize=(8, 5))
plt.barh(X.columns, importances)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.show()
